# 01 — EDA: Uzbek NER

Цель ноутбука — быстро понять, **что лежит в данных и какие риски важны для exact-span NER**.

Фокус:
- качество и баланс `ORG / NAME / GEO`;
- длина и составность сущностей;
- Latin / Cyrillic / mixed;
- апострофы и нестандартные символы;
- корректность `start/end`;
- overlap train/dev и доля unseen entities;
- автоматические выводы для следующих экспериментов.

> Ноутбук специально сделан компактным: только то, что влияет на модель, метрику и дальнейшие решения.


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 180)

# Если ноутбук лежит в notebooks/, оставь ROOT = Path("..")
# Если запускаешь из корня проекта — поменяй на Path(".")
ROOT = Path("..")
DATA_DIR = ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.jsonl"
DEV_PATH = DATA_DIR / "dev.jsonl"
MANIFEST_PATH = DATA_DIR / "dataset_manifest.json"

for p in [TRAIN_PATH, DEV_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Не найден {p}. Проверь ROOT/DATA_DIR и расположение train.jsonl/dev.jsonl."
        )

print("Train:", TRAIN_PATH.resolve())
print("Dev:  ", DEV_PATH.resolve())


## 1. Загрузка и проверка схемы

Сначала не строим графики, а смотрим, **как реально устроена одна запись**.  
Ноутбук пытается автоматически определить наиболее типичную схему (`text` + список сущностей со `start/end/type`).


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Некорректный JSON в {path}, строка {i}: {e}")
    return rows

train_raw = read_jsonl(TRAIN_PATH)
dev_raw = read_jsonl(DEV_PATH)

print(f"train rows: {len(train_raw):,}")
print(f"dev rows:   {len(dev_raw):,}")

print("\nПример train:")
display(train_raw[0])

if MANIFEST_PATH.exists():
    print("\nManifest:")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        display(json.load(f))
else:
    print("\ndataset_manifest.json не найден — это не блокирует EDA.")


In [ ]:
def infer_key(d, candidates):
    for k in candidates:
        if k in d:
            return k
    return None

sample = train_raw[0]

TEXT_KEY = infer_key(sample, ["text", "content", "sentence", "message"])
ENTITIES_KEY = infer_key(sample, ["entities", "spans", "labels", "annotations"])

print("Detected TEXT_KEY    =", TEXT_KEY)
print("Detected ENTITIES_KEY=", ENTITIES_KEY)

if TEXT_KEY is None or ENTITIES_KEY is None:
    raise KeyError(
        "Автоопределение схемы не сработало. "
        "Посмотри пример записи выше и вручную задай TEXT_KEY и ENTITIES_KEY в этой ячейке."
    )

first_entities = sample.get(ENTITIES_KEY, [])
print("\nПример сущности:")
display(first_entities[0] if first_entities else "В первой записи нет сущностей")


In [ ]:
# При необходимости поправь кандидаты под реальную схему одной сущности
def infer_entity_keys(entity):
    if not entity:
        return None, None, None, None
    start_key = infer_key(entity, ["start", "start_char", "begin", "offset_start"])
    end_key   = infer_key(entity, ["end", "end_char", "stop", "offset_end"])
    type_key  = infer_key(entity, ["type", "label", "entity_type", "tag"])
    text_key  = infer_key(entity, ["text", "value", "entity"])
    return start_key, end_key, type_key, text_key

probe_entity = next((e for r in train_raw for e in r.get(ENTITIES_KEY, []) if e), None)
START_KEY, END_KEY, TYPE_KEY, ENTITY_TEXT_KEY = infer_entity_keys(probe_entity)

print("START_KEY      =", START_KEY)
print("END_KEY        =", END_KEY)
print("TYPE_KEY       =", TYPE_KEY)
print("ENTITY_TEXT_KEY=", ENTITY_TEXT_KEY)

if None in [START_KEY, END_KEY, TYPE_KEY]:
    raise KeyError(
        "Не удалось определить поля start/end/type. "
        "Посмотри формат сущности выше и задай START_KEY/END_KEY/TYPE_KEY вручную."
    )


## 2. Нормализация в две таблицы

Получаем:
- `docs` — один ряд = один текст;
- `ents` — один ряд = одна сущность.

Это упростит все следующие проверки.


In [ ]:
def normalize_split(rows, split):
    docs = []
    ents = []

    for idx, r in enumerate(rows):
        text = r.get(TEXT_KEY, "")
        doc_id = r.get("id", r.get("text_id", r.get("uid", f"{split}_{idx}")))
        entities = r.get(ENTITIES_KEY, []) or []

        docs.append({
            "split": split,
            "doc_id": doc_id,
            "text": text,
            "n_chars": len(text),
            "n_entities": len(entities),
        })

        for j, e in enumerate(entities):
            start = e.get(START_KEY)
            end = e.get(END_KEY)
            label = e.get(TYPE_KEY)
            entity_text = e.get(ENTITY_TEXT_KEY) if ENTITY_TEXT_KEY else None

            span_text = None
            if isinstance(start, int) and isinstance(end, int) and 0 <= start <= end <= len(text):
                span_text = text[start:end]

            ents.append({
                "split": split,
                "doc_id": doc_id,
                "entity_idx": j,
                "label": label,
                "start": start,
                "end": end,
                "entity_text_annot": entity_text,
                "entity_text_span": span_text,
                "span_len_chars": (end - start) if isinstance(start, int) and isinstance(end, int) else np.nan,
                "span_len_words": len(span_text.split()) if isinstance(span_text, str) and span_text else 0,
            })

    return pd.DataFrame(docs), pd.DataFrame(ents)

train_docs, train_ents = normalize_split(train_raw, "train")
dev_docs, dev_ents = normalize_split(dev_raw, "dev")

docs = pd.concat([train_docs, dev_docs], ignore_index=True)
ents = pd.concat([train_ents, dev_ents], ignore_index=True)

display(docs.head())
display(ents.head())


## 3. Базовый профиль датасета

На что смотрим:
- размер train/dev;
- число сущностей;
- доля текстов без сущностей;
- длина текстов;
- среднее число сущностей на сообщение.

Если dev сильно отличается от train, validation может отражать не только качество модели, но и **distribution shift**.


In [ ]:
overview = (
    docs.groupby("split")
    .agg(
        texts=("doc_id", "count"),
        avg_chars=("n_chars", "mean"),
        median_chars=("n_chars", "median"),
        avg_entities=("n_entities", "mean"),
        texts_without_entities=("n_entities", lambda s: int((s == 0).sum())),
    )
)

entity_counts = ents.groupby("split").size().rename("entities")
overview = overview.join(entity_counts)
overview["share_without_entities"] = (
    overview["texts_without_entities"] / overview["texts"]
)

display(overview.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for split, g in docs.groupby("split"):
    ax.hist(g["n_chars"], bins=40, alpha=0.5, label=split)
ax.set_title("Распределение длины текстов")
ax.set_xlabel("Символов")
ax.set_ylabel("Количество текстов")
ax.legend()
plt.show()

display(
    docs.groupby("split")["n_chars"]
    .quantile([0.5, 0.9, 0.95, 0.99, 1.0])
    .unstack(0)
    .round(0)
)


## 4. Баланс `ORG / NAME / GEO`

Ключевой вопрос: нет ли класса, который сильно недопредставлен.  
Для exact-span micro-F1 большой класс сильнее влияет на итоговый score, но слабый F1 по редкому классу всё равно важен для бизнес-ценности.


In [ ]:
class_counts = (
    ents.groupby(["split", "label"])
    .size()
    .rename("count")
    .reset_index()
)

class_counts["share"] = class_counts.groupby("split")["count"].transform(lambda s: s / s.sum())
display(class_counts.sort_values(["split", "count"], ascending=[True, False]))

pivot = class_counts.pivot(index="label", columns="split", values="count").fillna(0)
pivot.plot(kind="bar", figsize=(8, 4))
plt.title("Распределение классов")
plt.xlabel("Класс")
plt.ylabel("Количество сущностей")
plt.xticks(rotation=0)
plt.show()


## 5. Длина и составность сущностей

Для нашего кейса особенно важны **длинные ORG/GEO**: чем больше слов в span, тем выше вероятность ошибки границ.


In [ ]:
entity_len_stats = (
    ents.groupby(["split", "label"])
    .agg(
        n=("entity_idx", "count"),
        median_chars=("span_len_chars", "median"),
        p90_chars=("span_len_chars", lambda s: s.quantile(0.9)),
        median_words=("span_len_words", "median"),
        p90_words=("span_len_words", lambda s: s.quantile(0.9)),
        multiword_share=("span_len_words", lambda s: (s > 1).mean()),
    )
    .round(3)
)

display(entity_len_stats)


In [ ]:
longest = (
    ents.dropna(subset=["entity_text_span"])
    .sort_values("span_len_chars", ascending=False)
    [["split", "label", "span_len_chars", "span_len_words", "entity_text_span"]]
    .head(25)
)
display(longest)


## 6. Latin / Cyrillic / mixed

Грубая script-эвристика — не полноценное language detection, но для EDA этого достаточно.

Важно понять:
- сколько чистой латиницы;
- сколько кириллицы;
- сколько mixed;
- отличается ли это между train/dev.

Если mixed/Cyrillic заметная доля, Latin-only Uzbek encoder может быть рискованным выбором.


In [ ]:
CYR_RE = re.compile(r"[А-Яа-яЁёЎўҚқҒғҲҳ]")
LAT_RE = re.compile(r"[A-Za-zʻʼ’`']")

def script_bucket(text):
    has_cyr = bool(CYR_RE.search(text))
    has_lat = bool(LAT_RE.search(text))
    if has_cyr and has_lat:
        return "mixed"
    if has_cyr:
        return "cyrillic"
    if has_lat:
        return "latin"
    return "other"

docs["script"] = docs["text"].map(script_bucket)

script_dist = (
    docs.groupby(["split", "script"])
    .size()
    .rename("count")
    .reset_index()
)
script_dist["share"] = script_dist.groupby("split")["count"].transform(lambda s: s / s.sum())

display(script_dist)

script_dist.pivot(index="script", columns="split", values="share").fillna(0).plot(
    kind="bar", figsize=(8, 4)
)
plt.title("Доля текстов по письменности")
plt.ylabel("Доля")
plt.xlabel("")
plt.xticks(rotation=0)
plt.show()


## 7. Апострофы и нестандартные варианты

Для узбекской латиницы это потенциально важный источник tokenization noise:

`O'zbekiston`, `O‘zbekiston`, `Oʻzbekiston`, ``O`zbekiston``

Если вариантов много, стоит отдельно проверить **normalization + mapping offsets обратно в исходный текст**.


In [ ]:
APOSTROPHES = ["'", "’", "ʻ", "ʼ", "`", "‘"]

rows = []
for split, g in docs.groupby("split"):
    total_chars = sum(len(t) for t in g["text"])
    for ch in APOSTROPHES:
        count = sum(t.count(ch) for t in g["text"])
        rows.append({
            "split": split,
            "char": repr(ch),
            "count": count,
            "per_10k_chars": count / max(total_chars, 1) * 10_000,
        })

apostrophe_df = pd.DataFrame(rows)
display(apostrophe_df.sort_values(["split", "count"], ascending=[True, False]))


## 8. Аудит разметки и exact spans

Это самый критичный раздел для хакатона.

Проверяем:
- `start/end` существуют и являются целыми;
- `0 <= start < end <= len(text)`;
- если в annotation хранится текст сущности — он совпадает с `text[start:end]`;
- нет дублей;
- нет overlapping spans.

Любая системная проблема здесь напрямую бьёт по exact-span F1.


In [ ]:
def audit_split(raw_rows, split):
    issues = []

    for idx, r in enumerate(raw_rows):
        text = r.get(TEXT_KEY, "")
        doc_id = r.get("id", r.get("text_id", r.get("uid", f"{split}_{idx}")))
        entities = r.get(ENTITIES_KEY, []) or []

        spans = []
        seen = set()

        for j, e in enumerate(entities):
            start = e.get(START_KEY)
            end = e.get(END_KEY)
            label = e.get(TYPE_KEY)

            if not isinstance(start, int) or not isinstance(end, int):
                issues.append((split, doc_id, j, "non_integer_offsets", start, end, label))
                continue

            if not (0 <= start < end <= len(text)):
                issues.append((split, doc_id, j, "invalid_offsets", start, end, label))
                continue

            key = (start, end, label)
            if key in seen:
                issues.append((split, doc_id, j, "duplicate_span", start, end, label))
            seen.add(key)

            span_text = text[start:end]
            if ENTITY_TEXT_KEY and e.get(ENTITY_TEXT_KEY) is not None:
                if e.get(ENTITY_TEXT_KEY) != span_text:
                    issues.append((split, doc_id, j, "text_span_mismatch", start, end, label))

            spans.append((start, end, label, j))

        spans = sorted(spans)
        for a, b in zip(spans, spans[1:]):
            if b[0] < a[1]:
                issues.append((split, doc_id, f"{a[3]}->{b[3]}", "overlap", b[0], a[1], f"{a[2]} / {b[2]}"))

    return pd.DataFrame(
        issues,
        columns=["split", "doc_id", "entity_idx", "issue", "start", "end", "label"]
    )

audit = pd.concat(
    [audit_split(train_raw, "train"), audit_split(dev_raw, "dev")],
    ignore_index=True
)

if audit.empty:
    print("✅ Явных проблем разметки не найдено.")
else:
    display(audit["issue"].value_counts().to_frame("count"))
    display(audit.head(30))


## 9. Train / dev overlap

Проверяем два разных эффекта:

1. **Duplicate leakage** — одинаковые тексты одновременно в train и dev.
2. **Entity memorization** — насколько часто сущности dev уже встречались в train.

Высокий overlap сущностей не обязательно плох, но тогда validation частично измеряет запоминание известных названий.  
Низкий overlap — наоборот, сильнее проверяет способность находить **новые сущности по контексту**.


In [ ]:
train_texts = set(train_docs["text"])
dev_texts = set(dev_docs["text"])
text_overlap = train_texts & dev_texts

print(f"Exact duplicate texts train↔dev: {len(text_overlap):,}")
print(f"Share of dev duplicated in train: {len(text_overlap) / max(len(dev_texts), 1):.2%}")


In [ ]:
def canonical_entity(s):
    if not isinstance(s, str):
        return None
    return re.sub(r"\s+", " ", s.strip().lower())

train_entity_set = set(
    train_ents["entity_text_span"].dropna().map(canonical_entity)
)
dev_entity_series = dev_ents["entity_text_span"].dropna().map(canonical_entity)

dev_seen = dev_entity_series.isin(train_entity_set)

print(f"Dev entities seen in train:   {dev_seen.mean():.2%}")
print(f"Dev entities unseen in train: {(~dev_seen).mean():.2%}")

seen_by_class = (
    dev_ents.assign(
        canonical=dev_ents["entity_text_span"].map(canonical_entity)
    )
    .dropna(subset=["canonical"])
    .assign(seen_in_train=lambda d: d["canonical"].isin(train_entity_set))
    .groupby("label")["seen_in_train"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "seen_share"})
)

display(seen_by_class.round(3))


## 10. Частые сущности и неоднозначные строки

Полезно посмотреть:
- какие сущности доминируют в выборке;
- встречается ли одна и та же строка с разными классами.

Второе особенно важно для `ORG ↔ GEO` и контекстно-зависимых случаев.


In [ ]:
freq = (
    train_ents.dropna(subset=["entity_text_span"])
    .groupby(["label", "entity_text_span"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)

display(freq.head(30))

ambiguous = (
    ents.dropna(subset=["entity_text_span"])
    .assign(canonical=lambda d: d["entity_text_span"].map(canonical_entity))
    .groupby("canonical")
    .agg(
        n_labels=("label", "nunique"),
        labels=("label", lambda s: sorted(set(s))),
        count=("label", "size"),
    )
    .query("n_labels > 1")
    .sort_values(["count", "n_labels"], ascending=False)
)

print(f"Строк с несколькими классами: {len(ambiguous):,}")
display(ambiguous.head(25))


## 11. Автоматические выводы для следующих экспериментов

Это не «истина», а быстрый список сигналов, которые стоит проверить на baseline/error analysis.


In [ ]:
findings = []

# 1. Class imbalance
train_cc = train_ents["label"].value_counts(normalize=True)
if len(train_cc) > 1 and train_cc.min() < 0.2:
    rare = train_cc.idxmin()
    findings.append({
        "Наблюдение": f"Класс {rare} занимает только {train_cc.min():.1%} train entities",
        "Риск": "Класс может иметь хуже Recall/F1",
        "Что проверить": "Per-class F1; ошибки FN; не оптимизировать только micro-F1"
    })

# 2. Multiword
multi = train_ents.groupby("label")["span_len_words"].apply(lambda s: (s > 1).mean())
if len(multi) and multi.max() > 0.35:
    lbl = multi.idxmax()
    findings.append({
        "Наблюдение": f"{lbl}: {multi.max():.1%} сущностей multi-token",
        "Риск": "Больше ошибок exact boundaries",
        "Что проверить": "Boundary errors по длине сущности; BIO decoding/postprocessing"
    })

# 3. Mixed / Cyrillic
script_share = train_docs["script"].value_counts(normalize=True)
non_latin = script_share.get("mixed", 0) + script_share.get("cyrillic", 0)
if non_latin > 0.1:
    findings.append({
        "Наблюдение": f"{non_latin:.1%} train текстов содержат Cyrillic/mixed",
        "Риск": "Latin-only encoder может терять качество",
        "Что проверить": "XLM-R vs Uzbek-only encoder; F1 по script-сегментам"
    })

# 4. Apostrophes
apost_counts = {ch: sum(t.count(ch) for t in train_docs["text"]) for ch in APOSTROPHES}
used_variants = sum(v > 0 for v in apost_counts.values())
if used_variants >= 3:
    findings.append({
        "Наблюдение": f"В train встречается {used_variants} вариантов апострофа",
        "Риск": "Фрагментация токенизации и разные surface forms одной сущности",
        "Что проверить": "Apostrophe normalization + сохранение mapping к исходным offsets"
    })

# 5. Unseen
unseen_share = (~dev_seen).mean() if len(dev_seen) else np.nan
if pd.notna(unseen_share) and unseen_share > 0.3:
    findings.append({
        "Наблюдение": f"{unseen_share:.1%} dev entity mentions не встречались в train",
        "Риск": "Validation требует генерализации, а не только запоминания",
        "Что проверить": "Seen vs unseen entity F1; pretrained multilingual encoders"
    })

# 6. Audit
if not audit.empty:
    findings.append({
        "Наблюдение": f"Найдено {len(audit)} потенциальных проблем разметки",
        "Риск": "Шум напрямую снижает потолок exact-span F1",
        "Что проверить": "Ручной просмотр audit-выборки и правила LABELING_GUIDE"
    })

# 7. Duplicates
if len(text_overlap) > 0:
    findings.append({
        "Наблюдение": f"{len(text_overlap)} exact duplicate texts между train и dev",
        "Риск": "Validation может быть оптимистичнее реальной генерализации",
        "Что проверить": "Метрики отдельно на duplicated/non-duplicated dev"
    })

findings_df = pd.DataFrame(findings)
if findings_df.empty:
    print("Явных автоматических красных флагов не найдено — переходи к ручному просмотру и baseline.")
else:
    display(findings_df)


# Что особенно важно проверить после запуска

После первого полного прогона EDA я бы в первую очередь посмотрел на пять вещей:

1. **ORG vs остальные классы**  
   Если ORG заметно длиннее и чаще multi-token — это главный кандидат на boundary bottleneck.

2. **Latin / Cyrillic / mixed distribution**  
   Это напрямую определяет, насколько оправдан monolingual Uzbek encoder против XLM-R / другого multilingual encoder.

3. **Апострофы**  
   Если одновременно встречаются `'`, `’`, `ʻ`, `ʼ`, `` ` ``, обязательно сделать отдельный tokenizer-анализ до эксперимента с normalization.

4. **Unseen entities в dev**  
   Если их много, модель должна хорошо обобщать по контексту. Простое memorization / словари будут иметь ограниченный потолок.

5. **Ошибки разметки / неоднозначные boundaries**  
   Для exact-span scoring это принципиальнее, чем для обычной token-level метрики. Все подозрительные кейсы надо сверять с `LABELING_GUIDE.md`.

## Что НЕ делать на этом этапе

- не делать десятки декоративных графиков;
- не начинать augmentation до понимания distribution;
- не нормализовать исходный текст без стратегии возврата координат;
- не выбирать модель только по опубликованному Uzbek NER F1 на другом датасете;
- не считать итоговый micro-F1 достаточным без breakdown по классам и script-сегментам.

Следующий шаг после этого ноутбука: **baseline + error analysis на тех же сегментах**.
